In [0]:
%sql
SHOW TABLES IN xliidw_uat_spl.staging_genius;

DESCRIBE HISTORY xliidw_uat_spl.staging_genius.t_genius_count


version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2025-10-28T14:12:28Z,5844722678443465,chirag.pundir@contractor.axaxl.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(119511364130134),0602-141703-v7wmxkv9,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 3176)",null,Databricks-Runtime/16.4.x-scala2.12
3,2025-10-21T14:03:11Z,145675729335058,ba859ef1-1dc0-47f3-8122-27d4936d2a1b,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])","List(327865792785246, Untitled, null, 885080678111000, 145675729335058, manual)",List(1648871177706507),1021-135733-vl4pu994,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 3193)",null,Databricks-Runtime/15.4.x-scala2.12
2,2025-10-09T07:15:59Z,5844722678443465,chirag.pundir@contractor.axaxl.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(995793994956718),0807-192317-sy2z670g,1,WriteSerializable,false,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 3360)",null,Databricks-Runtime/16.4.x-scala2.12
1,2025-10-08T09:26:23Z,145675729335058,ba859ef1-1dc0-47f3-8122-27d4936d2a1b,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])","List(693734791121849, Untitled, null, 1052938753887273, 145675729335058, manual)",List(1648871177706507),1008-084736-5aum901y,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 3193)",null,Databricks-Runtime/15.4.x-scala2.12
0,2025-09-30T09:57:08Z,145675729335058,ba859ef1-1dc0-47f3-8122-27d4936d2a1b,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)","List(222985522821718, Untitled, null, 288755658058964, 145675729335058, manual)",List(1648871177706507),0930-094700-b5ph1mel,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 3221)",null,Databricks-Runtime/15.4.x-scala2.12


In [0]:
df_lpl = spark.sql("SHOW TABLES IN xliidw_uat_lpl.staging_genius")
df_spl = spark.sql("SHOW TABLES IN xliidw_uat_spl.staging_genius")
lpl_table_list = [f"xliidw_uat_lpl.staging_genius.{row.tableName}" for row in df_lpl.collect()]
spl_table_list = [f"xliidw_uat_spl.staging_genius.{row.tableName}" for row in df_spl.collect()]
full_table_list = lpl_table_list + spl_table_list

In [0]:
print(full_table_list)

['xliidw_uat_lpl.staging_genius.accountrevenuelink', 'xliidw_uat_lpl.staging_genius.accountsheader', 'xliidw_uat_lpl.staging_genius.accountsheaderlink', 'xliidw_uat_lpl.staging_genius.accruedpremium', 'xliidw_uat_lpl.staging_genius.accruedpremium_deleted_rows', 'xliidw_uat_lpl.staging_genius.accruedpremium_holding', 'xliidw_uat_lpl.staging_genius.accruedpremium_holding_deleted_rows', 'xliidw_uat_lpl.staging_genius.accruedpremium_holding_monthly', 'xliidw_uat_lpl.staging_genius.accruedpremium_holding_monthly_deleted_rows', 'xliidw_uat_lpl.staging_genius.accruedpremium_incremental', 'xliidw_uat_lpl.staging_genius.accruedpremium_incremental_deleted_rows', 'xliidw_uat_lpl.staging_genius.addresscodes', 'xliidw_uat_lpl.staging_genius.addresscodes_deleted_rows', 'xliidw_uat_lpl.staging_genius.addressheader', 'xliidw_uat_lpl.staging_genius.addressheader_deleted_rows', 'xliidw_uat_lpl.staging_genius.aircraftdetails', 'xliidw_uat_lpl.staging_genius.allocationhistory', 'xliidw_uat_lpl.staging_gen

In [0]:
from pyspark.sql import functions as F

final_df = None

for table_name in full_table_list:
    try:
        history_df = spark.sql(f"DESCRIBE HISTORY {table_name}")
    except Exception:
        continue

    # Find the most recent CREATE OR REPLACE TABLE AS SELECT operation
    crt_ts = history_df.filter(
        F.col("operation") == "CREATE OR REPLACE TABLE AS SELECT"
    ).agg(
        F.max("timestamp").alias("max_ts")
    ).collect()[0]["max_ts"]
    #crt_ts="2025-01-20T08:10:59"

    # Filter for MERGE and WRITE after that timestamp and count
    result_df = history_df.filter(
    (F.col("timestamp") > crt_ts) &
    (F.col("operation").isin("MERGE", "WRITE"))
).select(
    F.lit(table_name).alias("table_name"),
    F.col("operation"),
    F.col("timestamp"),
    F.col("operationMetrics")["numTargetRowsInserted"].cast("long").alias("insertrow"),
    F.col("operationMetrics")["numTargetRowsUpdated"].cast("long").alias("updatedrow"),
    F.lit(crt_ts).alias("ingestion_dt")
) 

    if final_df is None:
        final_df = result_df
    else:
        final_df = final_df.unionByName(result_df)

display(final_df)

table_name,operation,timestamp,insertrow,updatedrow,ingestion_dt
xliidw_uat_lpl.staging_genius.accountrevenuelink,MERGE,2025-11-06T10:20:57Z,0,0,2025-10-20T05:51:42Z
xliidw_uat_lpl.staging_genius.accountrevenuelink,MERGE,2025-11-06T09:51:38Z,0,0,2025-10-20T05:51:42Z
xliidw_uat_lpl.staging_genius.accountrevenuelink,MERGE,2025-10-21T14:33:57Z,0,221206469,2025-10-20T05:51:42Z
xliidw_uat_lpl.staging_genius.accountsheader,MERGE,2025-10-21T08:50:04Z,0,431570,2025-10-20T05:51:58Z
xliidw_uat_lpl.staging_genius.accountsheaderlink,MERGE,2025-10-21T08:51:15Z,0,0,2025-10-20T05:52:06Z
xliidw_uat_lpl.staging_genius.accountsheaderlink,MERGE,2025-10-21T07:16:26Z,0,436738,2025-10-20T05:52:06Z
xliidw_uat_lpl.staging_genius.accruedpremium_holding,WRITE,2025-11-04T07:45:17Z,null,null,2025-11-04T06:57:34Z
xliidw_uat_lpl.staging_genius.addresscodes,MERGE,2025-10-21T11:09:15Z,0,6594964,2025-10-20T06:03:34Z
xliidw_uat_lpl.staging_genius.addressheader,MERGE,2025-10-21T11:11:08Z,0,7237972,2025-10-20T06:04:04Z
xliidw_uat_lpl.staging_genius.allocationhistory,MERGE,2025-11-06T06:26:14Z,0,15,2025-10-20T06:06:52Z


In [0]:
%sql
select count(1)
from xliidw_uat_lpl.staging_genius.alltransfulldetail

count(1)
303388293


In [0]:
%sql
select count(1)
from xliidw_uat_lpl.staging_xlprof_genius.alltransfulldetail

count(1)
916254


303388293  - swiss genius
916254    - swiss xlprof genius

